## Outlier Detection

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import seaborn as sns
from pathlib import Path
import json

class OutlierDetector:
    """
    Interactive outlier detection for identifying special/outlier real data points
    that synthetic generation might be missing
    """
    
    def __init__(self, embeddings_file, original_data_file, add_unique_ids=True):
        """
        Initialize with embeddings and original data
        
        Args:
            embeddings_file: Path to pickled embeddings file
            original_data_file: Path to original data CSV
            add_unique_ids: Whether to add unique IDs for tracking
        """
        print("🔍 Initializing Outlier Detector...")
        
        # Load embeddings
        with open(embeddings_file, 'rb') as f:
            self.embedding_data = pickle.load(f)
        
        self.embeddings = self.embedding_data['embeddings']
        self.types = self.embedding_data['types']
        self.indices = self.embedding_data['indices']
        
        # Load original data
        self.original_data = pd.read_csv(original_data_file, compression='gzip')
        
        # Handle unique IDs
        self.add_unique_ids = add_unique_ids
        if add_unique_ids:
            if 'unique_id' not in self.original_data.columns:
                print("🏷️  Adding unique IDs for data tracking...")
                self.original_data['unique_id'] = ['REAL_' + str(i).zfill(6) for i in range(len(self.original_data))]
                
                # Save updated data with unique IDs
                output_path = Path(original_data_file).parent / "annotated_data"
                output_path.mkdir(exist_ok=True)
                annotated_file = output_path / f"annotated_{Path(original_data_file).name}"
                self.original_data.to_csv(annotated_file, compression='gzip', index=False)
                print(f"💾 Saved annotated data with unique IDs: {annotated_file}")
                self.annotated_file_path = annotated_file
            else:
                print("✅ Unique IDs already present in data")
                self.annotated_file_path = original_data_file
        
        # Filter to only real data points
        self.real_mask = self.types == 'real'
        self.real_embeddings = self.embeddings[self.real_mask]
        self.real_indices = self.indices[self.real_mask]
        
        print(f"📊 Loaded {len(self.real_embeddings)} real data points")
        print(f"📊 Total embeddings: {len(self.embeddings)}")
        
    def prepare_dimensionality_reduction(self):
        """Prepare PCA and t-SNE for visualization"""
        print("🔄 Computing dimensionality reduction...")
        
        # Standardize embeddings
        scaler = StandardScaler()
        embeddings_scaled = scaler.fit_transform(self.embeddings)
        
        # PCA
        pca = PCA(n_components=2, random_state=42)
        self.pca_embeddings = pca.fit_transform(embeddings_scaled)
        
        # t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        self.tsne_embeddings = tsne.fit_transform(embeddings_scaled)
        
        # Store real data coordinates
        self.real_pca = self.pca_embeddings[self.real_mask]
        self.real_tsne = self.tsne_embeddings[self.real_mask]
        
        print("✅ Dimensionality reduction completed!")
        
    def detect_outliers_multiple_methods(self, contamination=0.1):
        """
        Detect outliers using multiple methods
        
        Args:
            contamination: Expected proportion of outliers
        """
        print("🎯 Detecting outliers using multiple methods...")
        
        outlier_methods = {}
        
        # Method 1: Distance-based outliers (far from synthetic data)
        synthetic_mask = self.types != 'real'
        if np.any(synthetic_mask):
            synthetic_embeddings = self.embeddings[synthetic_mask]
            synthetic_centroid = np.mean(synthetic_embeddings, axis=0)
            
            # Distance from real points to synthetic centroid
            distances_to_synthetic = cdist(self.real_embeddings, 
                                         synthetic_centroid.reshape(1, -1)).flatten()
            
            # Identify outliers as points far from synthetic centroid
            threshold = np.percentile(distances_to_synthetic, (1-contamination)*100)
            outlier_methods['distance_to_synthetic'] = distances_to_synthetic > threshold
        
        # Method 2: Isolation-based outliers using DBSCAN
        dbscan = DBSCAN(eps=0.5, min_samples=5)
        dbscan_labels = dbscan.fit_predict(self.real_embeddings)
        outlier_methods['dbscan'] = dbscan_labels == -1
        
        # Method 3: Statistical outliers in embedding space
        real_centroid = np.mean(self.real_embeddings, axis=0)
        distances_to_real_center = cdist(self.real_embeddings, 
                                       real_centroid.reshape(1, -1)).flatten()
        threshold = np.percentile(distances_to_real_center, (1-contamination)*100)
        outlier_methods['statistical'] = distances_to_real_center > threshold
        
        # Method 4: t-SNE based outliers (visual outliers)
        if hasattr(self, 'real_tsne'):
            tsne_centroid = np.mean(self.real_tsne, axis=0)
            tsne_distances = cdist(self.real_tsne, 
                                 tsne_centroid.reshape(1, -1)).flatten()
            tsne_threshold = np.percentile(tsne_distances, (1-contamination)*100)
            outlier_methods['tsne_visual'] = tsne_distances > tsne_threshold
        
        # Method 5: PCA based outliers
        if hasattr(self, 'real_pca'):
            pca_centroid = np.mean(self.real_pca, axis=0)
            pca_distances = cdist(self.real_pca, 
                                pca_centroid.reshape(1, -1)).flatten()
            pca_threshold = np.percentile(pca_distances, (1-contamination)*100)
            outlier_methods['pca_visual'] = pca_distances > pca_threshold
        
        self.outlier_methods = outlier_methods
        
        # Create consensus outliers (detected by multiple methods)
        outlier_votes = np.zeros(len(self.real_embeddings))
        for method, outliers in outlier_methods.items():
            outlier_votes += outliers.astype(int)
        
        # Consensus: detected by at least 2 methods
        self.consensus_outliers = outlier_votes >= 2
        self.outlier_votes = outlier_votes
        
        print(f"📈 Outlier detection summary:")
        for method, outliers in outlier_methods.items():
            print(f"  {method}: {np.sum(outliers)} outliers")
        print(f"  consensus (≥2 methods): {np.sum(self.consensus_outliers)} outliers")
        
        return outlier_methods
    
    def create_interactive_visualization(self, save_html=True):
        """
        Create interactive Plotly visualization for outlier exploration
        """
        print("📊 Creating interactive visualization...")
        
        # Prepare data for plotting
        plot_data = []
        
        # Add all data points
        for i, data_type in enumerate(['real', 'rewrite', 'rewrite_strong', 'rewrite_weak']):
            type_mask = self.types == data_type
            if not np.any(type_mask):
                continue
                
            type_data = {
                'x_pca': self.pca_embeddings[type_mask, 0],
                'y_pca': self.pca_embeddings[type_mask, 1],
                'x_tsne': self.tsne_embeddings[type_mask, 0],
                'y_tsne': self.tsne_embeddings[type_mask, 1],
                'type': data_type,
                'indices': self.indices[type_mask] if data_type == 'real' else [-1] * np.sum(type_mask)
            }
            plot_data.append(type_data)
        
        # Create subplots
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('PCA: All Data Types', 't-SNE: All Data Types', 
                          'PCA: Real Data Outliers', 't-SNE: Real Data Outliers'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        colors = ['blue', 'orange', 'green', 'red']
        
        # Plot 1 & 2: All data types
        for i, data in enumerate(plot_data):
            data_type = data['type']
            color = colors[i]
            
            # PCA plot
            fig.add_trace(
                go.Scatter(
                    x=data['x_pca'], y=data['y_pca'],
                    mode='markers',
                    name=f'{data_type}',
                    marker=dict(color=color, size=4, opacity=0.6),
                    text=[f'Type: {data_type}<br>Index: {idx}<br>ID: {self._get_unique_id(idx, data_type)}' for idx in data['indices']],
                    hovertemplate='%{text}<br>PCA1: %{x:.3f}<br>PCA2: %{y:.3f}<extra></extra>'
                ),
                row=1, col=1
            )
            
            # t-SNE plot
            fig.add_trace(
                go.Scatter(
                    x=data['x_tsne'], y=data['y_tsne'],
                    mode='markers',
                    name=f'{data_type}_tsne',
                    marker=dict(color=color, size=4, opacity=0.6),
                    text=[f'Type: {data_type}<br>Index: {idx}<br>ID: {self._get_unique_id(idx, data_type)}' for idx in data['indices']],
                    hovertemplate='%{text}<br>tSNE1: %{x:.3f}<br>tSNE2: %{y:.3f}<extra></extra>',
                    showlegend=False
                ),
                row=1, col=2
            )
        
        # Plot 3 & 4: Real data with outlier highlighting
        # Normal real data points
        normal_mask = ~self.consensus_outliers
        
        fig.add_trace(
            go.Scatter(
                x=self.real_pca[normal_mask, 0], 
                y=self.real_pca[normal_mask, 1],
                mode='markers',
                name='Normal Real Data',
                marker=dict(color='lightblue', size=6, opacity=0.6),
                text=[f'Normal Real<br>Index: {self.real_indices[i]}<br>ID: {self._get_unique_id(self.real_indices[i], "real")}<br>Votes: {self.outlier_votes[i]}' 
                      for i in range(len(normal_mask)) if normal_mask[i]],
                hovertemplate='%{text}<br>PCA1: %{x:.3f}<br>PCA2: %{y:.3f}<extra></extra>'
            ),
            row=2, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=self.real_tsne[normal_mask, 0], 
                y=self.real_tsne[normal_mask, 1],
                mode='markers',
                name='Normal Real Data (tSNE)',
                marker=dict(color='lightblue', size=6, opacity=0.6),
                text=[f'Normal Real<br>Index: {self.real_indices[i]}<br>ID: {self._get_unique_id(self.real_indices[i], "real")}<br>Votes: {self.outlier_votes[i]}' 
                      for i in range(len(normal_mask)) if normal_mask[i]],
                hovertemplate='%{text}<br>tSNE1: %{x:.3f}<br>tSNE2: %{y:.3f}<extra></extra>',
                showlegend=False
            ),
            row=2, col=2
        )
        
        # Outlier real data points
        outlier_mask = self.consensus_outliers
        
        fig.add_trace(
            go.Scatter(
                x=self.real_pca[outlier_mask, 0], 
                y=self.real_pca[outlier_mask, 1],
                mode='markers',
                name='Outlier Real Data',
                marker=dict(color='red', size=10, opacity=0.8, symbol='diamond'),
                text=[f'OUTLIER<br>Index: {self.real_indices[i]}<br>ID: {self._get_unique_id(self.real_indices[i], "real")}<br>Votes: {self.outlier_votes[i]}' 
                      for i in range(len(outlier_mask)) if outlier_mask[i]],
                hovertemplate='%{text}<br>PCA1: %{x:.3f}<br>PCA2: %{y:.3f}<extra></extra>'
            ),
            row=2, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=self.real_tsne[outlier_mask, 0], 
                y=self.real_tsne[outlier_mask, 1],
                mode='markers',
                name='Outlier Real Data (tSNE)',
                marker=dict(color='red', size=10, opacity=0.8, symbol='diamond'),
                text=[f'OUTLIER<br>Index: {self.real_indices[i]}<br>ID: {self._get_unique_id(self.real_indices[i], "real")}<br>Votes: {self.outlier_votes[i]}' 
                      for i in range(len(outlier_mask)) if outlier_mask[i]],
                hovertemplate='%{text}<br>tSNE1: %{x:.3f}<br>tSNE2: %{y:.3f}<extra></extra>',
                showlegend=False
            ),
            row=2, col=2
        )
        
        # Update layout
        fig.update_layout(
            title="Interactive Outlier Detection Dashboard",
            height=800,
            showlegend=True
        )
        
        # Update axis labels
        fig.update_xaxes(title_text="PCA1", row=1, col=1)
        fig.update_yaxes(title_text="PCA2", row=1, col=1)
        fig.update_xaxes(title_text="tSNE1", row=1, col=2)
        fig.update_yaxes(title_text="tSNE2", row=1, col=2)
        fig.update_xaxes(title_text="PCA1", row=2, col=1)
        fig.update_yaxes(title_text="PCA2", row=2, col=1)
        fig.update_xaxes(title_text="tSNE1", row=2, col=2)
        fig.update_yaxes(title_text="tSNE2", row=2, col=2)
        
        self.interactive_fig = fig
        
        if save_html:
            # Save to outlier_analysis directory
            output_path = Path("outlier_analysis")
            output_path.mkdir(exist_ok=True)
            html_path = output_path / "outlier_detection_dashboard.html"
            fig.write_html(html_path)
            print(f"💾 Interactive dashboard saved to: {html_path}")
        
        return fig
    
    def _get_unique_id(self, index, data_type):
        """Helper method to get unique ID for a data point"""
        if data_type == 'real' and self.add_unique_ids:
            if index < len(self.original_data):
                return self.original_data.iloc[index].get('unique_id', f'REAL_{index}')
        return f'{data_type.upper()}_{index}'
    
    def extract_outlier_data(self, output_dir="outlier_analysis"):
        """
        Extract and save outlier data for manual inspection
        """
        print("📋 Extracting outlier data...")
        
        # Create output directory
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Get outlier indices in original data
        outlier_original_indices = self.real_indices[self.consensus_outliers]
        
        # Extract outlier rows from original data
        outlier_data = self.original_data.iloc[outlier_original_indices].copy()
        
        # Add outlier analysis information
        outlier_data['outlier_votes'] = self.outlier_votes[self.consensus_outliers]
        outlier_data['outlier_methods'] = [
            ', '.join([method for method, detected in self.outlier_methods.items() 
                      if detected[i]]) 
            for i in range(len(self.consensus_outliers)) 
            if self.consensus_outliers[i]
        ]
        
        # Add embedding coordinates
        outlier_data['pca_1'] = self.real_pca[self.consensus_outliers, 0]
        outlier_data['pca_2'] = self.real_pca[self.consensus_outliers, 1]
        outlier_data['tsne_1'] = self.real_tsne[self.consensus_outliers, 0]
        outlier_data['tsne_2'] = self.real_tsne[self.consensus_outliers, 1]
        
        # Save outlier data
        outlier_csv_path = output_path / "outlier_real_data.csv"
        outlier_data.to_csv(outlier_csv_path, index=False)
        
        # Save outlier analysis summary
        summary = {
            'total_real_samples': len(self.real_embeddings),
            'outlier_samples': len(outlier_data),
            'outlier_percentage': len(outlier_data) / len(self.real_embeddings) * 100,
            'detection_methods': list(self.outlier_methods.keys()),
            'outlier_indices': outlier_original_indices.tolist(),
            'method_summary': {
                method: int(np.sum(detected)) 
                for method, detected in self.outlier_methods.items()
            }
        }
        
        summary_path = output_path / "outlier_analysis_summary.json"
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"📊 Outlier Analysis Summary:")
        print(f"  Total real samples: {summary['total_real_samples']}")
        print(f"  Outlier samples: {summary['outlier_samples']}")
        print(f"  Outlier percentage: {summary['outlier_percentage']:.2f}%")
        print(f"📁 Files saved to: {output_path}")
        print(f"  - {outlier_csv_path}")
        print(f"  - {summary_path}")
        
        return outlier_data, summary
    
    def create_sampling_seeds(self, include_normal_ratio=0.3, output_dir="sampling_seeds"):
        """
        Create sampling seeds that include outliers and some normal samples
        
        Args:
            include_normal_ratio: Ratio of normal samples to include with outliers
        """
        print("🌱 Creating sampling seeds for improved synthetic generation...")
        
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Get outlier samples
        outlier_indices = self.real_indices[self.consensus_outliers]
        outlier_samples = self.original_data.iloc[outlier_indices].copy()
        
        # Get some normal samples
        normal_indices = self.real_indices[~self.consensus_outliers]
        n_normal_samples = int(len(outlier_samples) * include_normal_ratio)
        
        if n_normal_samples > 0 and len(normal_indices) > 0:
            selected_normal_indices = np.random.choice(
                normal_indices, 
                size=min(n_normal_samples, len(normal_indices)), 
                replace=False
            )
            normal_samples = self.original_data.iloc[selected_normal_indices].copy()
            normal_samples['seed_type'] = 'normal'
        else:
            normal_samples = pd.DataFrame()
        
        # Mark outlier samples
        outlier_samples['seed_type'] = 'outlier'
        
        # Combine samples
        if len(normal_samples) > 0:
            seed_samples = pd.concat([outlier_samples, normal_samples], ignore_index=True)
        else:
            seed_samples = outlier_samples
        
        # Save seed samples
        seed_csv_path = output_path / "sampling_seeds.csv"
        seed_samples.to_csv(seed_csv_path, index=False)
        
        # Create seed metadata
        seed_metadata = {
            'total_seeds': len(seed_samples),
            'outlier_seeds': len(outlier_samples),
            'normal_seeds': len(normal_samples) if len(normal_samples) > 0 else 0,
            'outlier_ratio': len(outlier_samples) / len(seed_samples),
            'generation_purpose': 'focused_sampling_for_synthetic_generation',
            'recommendation': 'Use these seeds to generate synthetic data that covers outlier patterns'
        }
        
        metadata_path = output_path / "seed_metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(seed_metadata, f, indent=2)
        
        print(f"🌱 Sampling seeds created:")
        print(f"  Total seeds: {seed_metadata['total_seeds']}")
        print(f"  Outlier seeds: {seed_metadata['outlier_seeds']}")
        print(f"  Normal seeds: {seed_metadata['normal_seeds']}")
        print(f"  Outlier ratio: {seed_metadata['outlier_ratio']:.2f}")
        print(f"📁 Files saved to: {output_path}")
        
        return seed_samples, seed_metadata

def run_outlier_analysis(embeddings_file, original_data_file, contamination=0.1, add_unique_ids=True):
    """
    Complete outlier analysis pipeline
    
    Args:
        embeddings_file: Path to embeddings pickle file
        original_data_file: Path to original data CSV
        contamination: Expected proportion of outliers (0.1 = 10%)
        add_unique_ids: Whether to add unique IDs for tracking
    """
    print("🚀 Starting outlier analysis pipeline...")
    
    # Initialize detector
    detector = OutlierDetector(embeddings_file, original_data_file, add_unique_ids=add_unique_ids)
    
    # Prepare visualizations
    detector.prepare_dimensionality_reduction()
    
    # Detect outliers
    detector.detect_outliers_multiple_methods(contamination=contamination)
    
    # Create interactive visualization
    detector.create_interactive_visualization(save_html=True)
    
    # Extract outlier data
    outlier_data, summary = detector.extract_outlier_data()
    
    # Create sampling seeds
    seed_samples, seed_metadata = detector.create_sampling_seeds()
    
    print("✅ Outlier analysis completed!")
    print("\n📋 Next steps:")
    print("1. Open 'outlier_analysis/outlier_detection_dashboard.html' to explore outliers interactively")
    print("2. Review 'outlier_analysis/outlier_real_data.csv' to understand outlier patterns")
    print("3. Use 'sampling_seeds/sampling_seeds.csv' for targeted synthetic generation")
    if detector.add_unique_ids and hasattr(detector, 'annotated_file_path'):
        print(f"4. Updated original data with unique IDs saved to: {detector.annotated_file_path}")
    print("5. Re-run embedding analysis after generating new synthetic data")
    
    return detector, outlier_data, seed_samples

# Example usage
if __name__ == "__main__":
    # Configuration
    EMBEDDINGS_FILE = "../data/embedding/combined_CEAS-08_malicious_all-MiniLM-L6-v2_embeddings.pkl"
    ORIGINAL_DATA_FILE = "../raw/email_phishing_CEAS-08_train.csv.gz"
    
    # Run analysis
    detector, outlier_data, seed_samples = run_outlier_analysis(
        EMBEDDINGS_FILE, 
        ORIGINAL_DATA_FILE,
        contamination=0.15,  # Expect 15% outliers
        add_unique_ids=True  # Add unique IDs for tracking
    )

🚀 Starting outlier analysis pipeline...
🔍 Initializing Outlier Detector...
📊 Loaded 1000 real data points
📊 Total embeddings: 4000
🔄 Computing dimensionality reduction...
✅ Dimensionality reduction completed!
🎯 Detecting outliers using multiple methods...
📈 Outlier detection summary:
  distance_to_synthetic: 150 outliers
  dbscan: 754 outliers
  statistical: 150 outliers
  tsne_visual: 150 outliers
  pca_visual: 109 outliers
  consensus (≥2 methods): 317 outliers
📊 Creating interactive visualization...
💾 Interactive dashboard saved to: outlier_detection_dashboard.html
📋 Extracting outlier data...
📊 Outlier Analysis Summary:
  Total real samples: 1000
  Outlier samples: 317
  Outlier percentage: 31.70%
📁 Files saved to: outlier_analysis
  - outlier_analysis\outlier_real_data.csv
  - outlier_analysis\outlier_analysis_summary.json
🌱 Creating sampling seeds for improved synthetic generation...
🌱 Sampling seeds created:
  Total seeds: 412
  Outlier seeds: 317
  Normal seeds: 95
  Outlier ra